In [5]:
import os
import glob
import csv
import re
import pandas as pd
import hashlib


In [ ]:

# =====================================
# 1. Set paths
# =====================================
log_folder = r"D:\Downloads\Capstone\Project\HDFS_v2\node_logs"
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"


# for mac
#log_folder = r"D:\Downloads\Capstone\Project\HDFS_v2\node_logs"
#output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"


os.makedirs(output_dir, exist_ok=True)

print("Log folder:", log_folder)
print("Output folder:", output_dir)



In [8]:
# =====================================
# 1. Set paths
# =====================================
log_folder = r"D:\Downloads\Capstone\Project\HDFS_v2\node_logs"
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"


# for mac
#log_folder = r"D:\Downloads\Capstone\Project\HDFS_v2\node_logs"
#output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"


os.makedirs(output_dir, exist_ok=True)

print("Log folder:", log_folder)
print("Output folder:", output_dir)



Log folder: D:\Downloads\Capstone\Project\HDFS_v2\node_logs
Output folder: D:\Downloads\Capstone\Project\logOutput_v2


In [10]:
output_file = os.path.join(output_dir, "hdfs_v2_combined_raw_logs.csv")

# 3. Find log files
log_files = glob.glob(os.path.join(log_folder, "*"))
log_files = [f for f in log_files if os.path.isfile(f)]

print("Total log files found:", len(log_files))
print("\nFirst 10 files:")
for f in log_files[:10]:
    print(os.path.basename(f))
    


Total log files found: 31

First 10 files:
hadoop-hdfs-datanode-mesos-01.log
hadoop-hdfs-datanode-mesos-02.log
hadoop-hdfs-datanode-mesos-03.log
hadoop-hdfs-datanode-mesos-05.log
hadoop-hdfs-datanode-mesos-06.log
hadoop-hdfs-datanode-mesos-07.log
hadoop-hdfs-datanode-mesos-08.log
hadoop-hdfs-datanode-mesos-09.log
hadoop-hdfs-datanode-mesos-10.log
hadoop-hdfs-datanode-mesos-11.log


In [11]:
# 4. Clean bad text

def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\x00", "")
    text = re.sub(r"[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    return text.strip()

In [12]:
# 5. Stream logs into one CSV (OVERWRITE MODE)
# =====================================
total_written = 0
total_skipped = 0

with open(output_file, "w", newline="", encoding="utf-8-sig") as out_csv:
    writer = csv.writer(out_csv, quoting=csv.QUOTE_MINIMAL)

    # always write header because file is overwritten
    writer.writerow(["source_file", "line_number", "raw_log"])

    for i, file_path in enumerate(log_files, start=1):
        file_name = os.path.basename(file_path)
        print(f"\nProcessing {i}/{len(log_files)}: {file_name}")

        file_written = 0
        file_skipped = 0

        try:
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                for line_num, line in enumerate(f, start=1):
                    try:
                        line = clean_text(line)
                        if not line:
                            continue

                        writer.writerow([file_name, line_num, line])
                        total_written += 1
                        file_written += 1

                    except Exception:
                        total_skipped += 1
                        file_skipped += 1
                        continue

        except Exception as e:
            print(f"Failed file: {file_name} | Error: {e}")
            continue

        print(f"  written: {file_written}")
        print(f"  skipped: {file_skipped}")

print("\nDone.")
print("Total written:", total_written)
print("Total skipped:", total_skipped)
print("Saved to:", output_file)


Processing 1/31: hadoop-hdfs-datanode-mesos-01.log
  written: 2614806
  skipped: 0

Processing 2/31: hadoop-hdfs-datanode-mesos-02.log
  written: 1916188
  skipped: 0

Processing 3/31: hadoop-hdfs-datanode-mesos-03.log
  written: 2046864
  skipped: 0

Processing 4/31: hadoop-hdfs-datanode-mesos-05.log
  written: 1793138
  skipped: 0

Processing 5/31: hadoop-hdfs-datanode-mesos-06.log
  written: 1909487
  skipped: 0

Processing 6/31: hadoop-hdfs-datanode-mesos-07.log
  written: 2036700
  skipped: 0

Processing 7/31: hadoop-hdfs-datanode-mesos-08.log
  written: 1972478
  skipped: 0

Processing 8/31: hadoop-hdfs-datanode-mesos-09.log
  written: 1948842
  skipped: 0

Processing 9/31: hadoop-hdfs-datanode-mesos-10.log
  written: 1869808
  skipped: 0

Processing 10/31: hadoop-hdfs-datanode-mesos-11.log
  written: 1757406
  skipped: 0

Processing 11/31: hadoop-hdfs-datanode-mesos-13.log
  written: 1968173
  skipped: 0

Processing 12/31: hadoop-hdfs-datanode-mesos-14.log
  written: 1798901
  

In [13]:
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
output_file = os.path.join(output_dir, "hdfs_v2_combined_raw_logs.csv")

df_sample = pd.read_csv(output_file, nrows=10, encoding="utf-8-sig")
print(df_sample.shape)
print(df_sample.head(10))

(10, 3)
                         source_file  line_number  \
0  hadoop-hdfs-datanode-mesos-01.log            1   
1  hadoop-hdfs-datanode-mesos-01.log            2   
2  hadoop-hdfs-datanode-mesos-01.log            3   
3  hadoop-hdfs-datanode-mesos-01.log            4   
4  hadoop-hdfs-datanode-mesos-01.log            5   
5  hadoop-hdfs-datanode-mesos-01.log            6   
6  hadoop-hdfs-datanode-mesos-01.log            7   
7  hadoop-hdfs-datanode-mesos-01.log            8   
8  hadoop-hdfs-datanode-mesos-01.log            9   
9  hadoop-hdfs-datanode-mesos-01.log           10   

                                             raw_log  
0  2015-12-03 14:37:47,611 INFO org.apache.hadoop...  
1  /*********************************************...  
2                     STARTUP_MSG: Starting DataNode  
3   STARTUP_MSG:   host = mesos-master-1/10.10.34.11  
4                           STARTUP_MSG:   args = []  
5                     STARTUP_MSG:   version = 2.7.1  
6  STARTUP_MSG:   class

In [ ]:
#Merge multiline log entries into single records

# =====================================
# 1. Paths
# =====================================
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
input_file = os.path.join(output_dir, "hdfs_v2_combined_raw_logs.csv")
merged_file = os.path.join(output_dir, "hdfs_v2_merged_logs.csv")

# =====================================
# 2. Pattern for NEW log entry
# Example:
# 2015-12-03 14:37:47,611 INFO org.apache.hadoop.hdfs.server.datanode.DataNode: ...
# =====================================
new_entry_pattern = re.compile(
    r'^\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2},\d{3}\s+[A-Z]+\s+'
)

total_raw_rows = 0
merged_count = 0

with open(input_file, "r", encoding="utf-8-sig", newline="") as f_in:
    reader = csv.DictReader(f_in)

    with open(merged_file, "w", encoding="utf-8-sig", newline="") as f_out:
        writer = csv.writer(f_out)
        writer.writerow(["source_file", "start_line_number", "merged_log"])

        current_source = None
        current_start_line = None
        current_log_parts = []

        for row in reader:
            total_raw_rows += 1
            source_file = row["source_file"]
            line_number = row["line_number"]
            raw_log = str(row["raw_log"]).strip()

            # flush if source file changes
            if current_source is not None and source_file != current_source:
                if current_log_parts:
                    writer.writerow([
                        current_source,
                        current_start_line,
                        " ".join(current_log_parts)
                    ])
                    merged_count += 1

                current_source = None
                current_start_line = None
                current_log_parts = []

            # new proper log entry
            if new_entry_pattern.match(raw_log):
                if current_log_parts:
                    writer.writerow([
                        current_source,
                        current_start_line,
                        " ".join(current_log_parts)
                    ])
                    merged_count += 1

                current_source = source_file
                current_start_line = line_number
                current_log_parts = [raw_log]
            else:
                # continuation line
                if current_log_parts:
                    current_log_parts.append(raw_log)
                else:
                    # orphan line before any timestamped log
                    current_source = source_file
                    current_start_line = line_number
                    current_log_parts = [raw_log]

        # flush final record
        if current_log_parts:
            writer.writerow([
                current_source,
                current_start_line,
                " ".join(current_log_parts)
            ])
            merged_count += 1

print("Done merging.")
print("Total raw rows:", total_raw_rows)
print("Merged log entries:", merged_count)
print("Saved to:", merged_file)

Done merging.
Total raw rows: 71116785
Merged log entries: 58095614
Saved to: D:\Downloads\Capstone\Project\logOutput_v2\hdfs_v2_merged_logs.csv


In [15]:
# Parse merged records into structured columns
# =====================================
# 1. Paths
# =====================================
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
merged_file = os.path.join(output_dir, "hdfs_v2_merged_logs.csv")
parsed_file = os.path.join(output_dir, "hdfs_v2_structured_logs.csv")

# =====================================
# 2. Parse pattern
# Example:
# 2015-12-03 14:37:47,611 INFO org.apache.hadoop.hdfs.server.datanode.DataNode: STARTUP_MSG: ...
# =====================================
parse_pattern = re.compile(
    r'^(?P<Date>\d{4}-\d{2}-\d{2})\s+'
    r'(?P<Time>\d{2}:\d{2}:\d{2},\d{3})\s+'
    r'(?P<Level>[A-Z]+)\s+'
    r'(?P<Component>.*?):\s+'
    r'(?P<Content>.*)$'
)

block_pattern = re.compile(r'(blk_-?\d+)')

def parse_merged_log(text):
    if pd.isna(text):
        return None

    text = str(text).strip()
    m = parse_pattern.match(text)
    if not m:
        return None

    row = m.groupdict()
    block_match = block_pattern.search(row["Content"])
    row["BlockId"] = block_match.group(1) if block_match else None
    row["Content"] = row["Content"].strip()
    return row

chunk_size = 50000
first_chunk = True
total_rows = 0
parsed_rows = 0
failed_rows = 0

for chunk in pd.read_csv(merged_file, chunksize=chunk_size, encoding="utf-8-sig"):
    total_rows += len(chunk)
    rows_out = []

    for _, row in chunk.iterrows():
        parsed = parse_merged_log(row["merged_log"])
        if parsed:
            parsed["source_file"] = row["source_file"]
            parsed["start_line_number"] = row["start_line_number"]
            parsed["raw_log"] = row["merged_log"]
            rows_out.append(parsed)
        else:
            failed_rows += 1

    if rows_out:
        out_df = pd.DataFrame(rows_out)
        parsed_rows += len(out_df)

        out_df.to_csv(
            parsed_file,
            mode="w" if first_chunk else "a",
            index=False,
            header=first_chunk,
            encoding="utf-8-sig"
        )
        first_chunk = False

    print(f"Processed: {total_rows:,} | Parsed: {parsed_rows:,} | Failed: {failed_rows:,}")

print("\nDone parsing.")
print("Total merged rows:", total_rows)
print("Successfully parsed:", parsed_rows)
print("Failed to parse:", failed_rows)
print("Saved to:", parsed_file)

Processed: 50,000 | Parsed: 50,000 | Failed: 0
Processed: 100,000 | Parsed: 100,000 | Failed: 0
Processed: 150,000 | Parsed: 150,000 | Failed: 0
Processed: 200,000 | Parsed: 200,000 | Failed: 0
Processed: 250,000 | Parsed: 250,000 | Failed: 0
Processed: 300,000 | Parsed: 300,000 | Failed: 0
Processed: 350,000 | Parsed: 350,000 | Failed: 0
Processed: 400,000 | Parsed: 400,000 | Failed: 0
Processed: 450,000 | Parsed: 450,000 | Failed: 0
Processed: 500,000 | Parsed: 500,000 | Failed: 0
Processed: 550,000 | Parsed: 550,000 | Failed: 0
Processed: 600,000 | Parsed: 600,000 | Failed: 0
Processed: 650,000 | Parsed: 650,000 | Failed: 0
Processed: 700,000 | Parsed: 700,000 | Failed: 0
Processed: 750,000 | Parsed: 750,000 | Failed: 0
Processed: 800,000 | Parsed: 800,000 | Failed: 0
Processed: 850,000 | Parsed: 850,000 | Failed: 0
Processed: 900,000 | Parsed: 900,000 | Failed: 0
Processed: 950,000 | Parsed: 950,000 | Failed: 0
Processed: 1,000,000 | Parsed: 1,000,000 | Failed: 0
Processed: 1,050,0

In [17]:
# Preview the structured result

output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
parsed_file = os.path.join(output_dir, "hdfs_v2_structured_logs.csv")

sample_df = pd.read_csv(parsed_file, nrows=10, encoding="utf-8-sig")
print(sample_df.columns.tolist())
print(sample_df.head(10))

['Date', 'Time', 'Level', 'Component', 'Content', 'BlockId', 'source_file', 'start_line_number', 'raw_log']
         Date          Time Level  \
0  2015-12-03  14:37:47,611  INFO   
1  2015-12-03  14:37:47,618  INFO   
2  2015-12-03  14:37:48,253  INFO   
3  2015-12-03  14:37:48,315  INFO   
4  2015-12-03  14:37:48,315  INFO   
5  2015-12-03  14:37:48,319  INFO   
6  2015-12-03  14:37:48,321  INFO   
7  2015-12-03  14:37:48,329  INFO   
8  2015-12-03  14:37:48,354  INFO   
9  2015-12-03  14:37:48,356  INFO   

                                           Component  \
0    org.apache.hadoop.hdfs.server.datanode.DataNode   
1    org.apache.hadoop.hdfs.server.datanode.DataNode   
2      org.apache.hadoop.metrics2.impl.MetricsConfig   
3  org.apache.hadoop.metrics2.impl.MetricsSystemImpl   
4  org.apache.hadoop.metrics2.impl.MetricsSystemImpl   
5  org.apache.hadoop.hdfs.server.datanode.BlockSc...   
6    org.apache.hadoop.hdfs.server.datanode.DataNode   
7    org.apache.hadoop.hdfs.server.d

Create EventTemplate and EventId

In [6]:
#import hashlib

output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
input_file = os.path.join(output_dir, "hdfs_v2_structured_logs.csv")
output_file = os.path.join(output_dir, "hdfs_v2_event_templates_v2.csv")

# =====================================
# 2. Stronger normalization patterns
# =====================================
ip_port_pattern = re.compile(r'\b\d+\.\d+\.\d+\.\d+:\d+\b')
ip_pattern = re.compile(r'\b\d+\.\d+\.\d+\.\d+\b')
block_pattern = re.compile(r'blk_-?\d+')
hex_pattern = re.compile(r'\b0x[a-fA-F0-9]+\b')
uuid_like_pattern = re.compile(r'\b[a-f0-9]{8,}\b', re.IGNORECASE)
path_pattern = re.compile(r'(/[A-Za-z0-9_.\-]+(?:/[A-Za-z0-9_.\-]+)+)')
host_num_pattern = re.compile(r'([A-Za-z_\-]+)-\d+\b')
long_num_pattern = re.compile(r'\b\d{2,}\b')
standalone_num_pattern = re.compile(r'\b\d+\b')

def make_template(text):
    text = str(text)

    text = ip_port_pattern.sub('<IP_PORT>', text)
    text = ip_pattern.sub('<IP>', text)
    text = block_pattern.sub('<BLOCK>', text)
    text = hex_pattern.sub('<HEX>', text)
    text = uuid_like_pattern.sub('<ID>', text)
    text = path_pattern.sub('<PATH>', text)
    text = host_num_pattern.sub(r'\1-<NUM>', text)
    text = long_num_pattern.sub('<NUM>', text)
    text = standalone_num_pattern.sub('<NUM>', text)

    text = re.sub(r'\s+', ' ', text).strip()
    return text

def event_id_from_template(template):
    return hashlib.md5(template.encode("utf-8")).hexdigest()[:8]

# =====================================
# 3. Rebuild event templates safely
# =====================================
chunk_size = 50000
first_chunk = True
total_rows = 0

for chunk in pd.read_csv(input_file, chunksize=chunk_size, encoding="utf-8-sig"):
    total_rows += len(chunk)

    chunk["EventTemplate"] = chunk["Content"].astype(str).apply(make_template)
    chunk["EventId"] = chunk["EventTemplate"].apply(event_id_from_template)

    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        index=False,
        header=first_chunk,
        encoding="utf-8-sig"
    )
    first_chunk = False

    print(f"Processed rows: {total_rows:,}")

print("\nDone.")
print("Saved to:", output_file)

Processed rows: 50,000
Processed rows: 100,000
Processed rows: 150,000
Processed rows: 200,000
Processed rows: 250,000
Processed rows: 300,000
Processed rows: 350,000
Processed rows: 400,000
Processed rows: 450,000
Processed rows: 500,000
Processed rows: 550,000
Processed rows: 600,000
Processed rows: 650,000
Processed rows: 700,000
Processed rows: 750,000
Processed rows: 800,000
Processed rows: 850,000
Processed rows: 900,000
Processed rows: 950,000
Processed rows: 1,000,000
Processed rows: 1,050,000
Processed rows: 1,100,000
Processed rows: 1,150,000
Processed rows: 1,200,000
Processed rows: 1,250,000
Processed rows: 1,300,000
Processed rows: 1,350,000
Processed rows: 1,400,000
Processed rows: 1,450,000
Processed rows: 1,500,000
Processed rows: 1,550,000
Processed rows: 1,600,000
Processed rows: 1,650,000
Processed rows: 1,700,000
Processed rows: 1,750,000
Processed rows: 1,800,000
Processed rows: 1,850,000
Processed rows: 1,900,000
Processed rows: 1,950,000
Processed rows: 2,000,000

: 

Alternate for event template, shorter version

In [3]:
import os
import re
import hashlib
import pandas as pd

# =====================================
# 1. Paths
# =====================================
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
input_file = os.path.join(output_dir, "hdfs_v2_structured_logs.csv")
output_file = os.path.join(output_dir, "hdfs_v2_event_templates_v2.csv")

# =====================================
# 2. Tuned settings for manageable training
# =====================================
chunk_size = 50000
target_block_count = 50000   # good balance: enough sessions, much smaller than full data
random_state = 42

# =====================================
# 3. Regex patterns
# =====================================
ip_port_pattern = re.compile(r'\b\d+\.\d+\.\d+\.\d+:\d+\b')
ip_pattern = re.compile(r'\b\d+\.\d+\.\d+\.\d+\b')
block_pattern = re.compile(r'blk_-?\d+')
hex_pattern = re.compile(r'\b0x[a-fA-F0-9]+\b')
uuid_like_pattern = re.compile(r'\b[a-f0-9]{8,}\b', re.IGNORECASE)
path_pattern = re.compile(r'(/[A-Za-z0-9_.\-]+(?:/[A-Za-z0-9_.\-]+)+)')
host_num_pattern = re.compile(r'([A-Za-z_\-]+)-\d+\b')
long_num_pattern = re.compile(r'\b\d{2,}\b')
standalone_num_pattern = re.compile(r'\b\d+\b')

def make_template(text):
    text = str(text)
    text = ip_port_pattern.sub('<IP_PORT>', text)
    text = ip_pattern.sub('<IP>', text)
    text = block_pattern.sub('<BLOCK>', text)
    text = hex_pattern.sub('<HEX>', text)
    text = uuid_like_pattern.sub('<ID>', text)
    text = path_pattern.sub('<PATH>', text)
    text = host_num_pattern.sub(r'\1-<NUM>', text)
    text = long_num_pattern.sub('<NUM>', text)
    text = standalone_num_pattern.sub('<NUM>', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def event_id_from_template(template):
    return hashlib.md5(template.encode("utf-8")).hexdigest()[:8]

# =====================================
# 4. Pass 1: collect a manageable sample of BlockIds
# =====================================
print("Pass 1: collecting BlockIds...")

all_block_ids = set()

for chunk in pd.read_csv(
    input_file,
    usecols=["BlockId"],
    chunksize=chunk_size,
    encoding="utf-8-sig"
):
    chunk = chunk.dropna(subset=["BlockId"])
    all_block_ids.update(chunk["BlockId"].astype(str).unique())

print("Total unique BlockIds found:", len(all_block_ids))

all_block_ids = pd.Series(sorted(all_block_ids))
sampled_block_ids = set(
    all_block_ids.sample(
        n=min(target_block_count, len(all_block_ids)),
        random_state=random_state
    ).tolist()
)

print("Sampled BlockIds kept:", len(sampled_block_ids))

# =====================================
# 5. Pass 2: build smaller event-template file
# =====================================
print("\nPass 2: creating reduced event-template file...")

first_chunk = True
total_rows = 0
kept_rows = 0

for chunk in pd.read_csv(input_file, chunksize=chunk_size, encoding="utf-8-sig"):
    total_rows += len(chunk)

    chunk = chunk.dropna(subset=["BlockId"]).copy()
    if len(chunk) == 0:
        continue

    chunk["BlockId"] = chunk["BlockId"].astype(str)
    chunk = chunk[chunk["BlockId"].isin(sampled_block_ids)].copy()

    if len(chunk) == 0:
        continue

    chunk["EventTemplate"] = chunk["Content"].astype(str).apply(make_template)
    chunk["EventId"] = chunk["EventTemplate"].apply(event_id_from_template)

    # keep only needed columns for later steps
    chunk = chunk[[
        "Date", "Time", "Level", "Component",
        "Content", "BlockId", "EventTemplate", "EventId"
    ]]

    kept_rows += len(chunk)

    chunk.to_csv(
        output_file,
        mode="w" if first_chunk else "a",
        index=False,
        header=first_chunk,
        encoding="utf-8-sig"
    )
    first_chunk = False

    print(f"Processed rows: {total_rows:,} | Kept rows: {kept_rows:,}")

print("\nDone.")
print("Saved to:", output_file)
print("Total kept rows:", kept_rows)
print("Target BlockIds:", len(sampled_block_ids))

Pass 1: collecting BlockIds...
Total unique BlockIds found: 2460805
Sampled BlockIds kept: 50000

Pass 2: creating reduced event-template file...
Processed rows: 50,000 | Kept rows: 944
Processed rows: 100,000 | Kept rows: 1,862
Processed rows: 150,000 | Kept rows: 2,852
Processed rows: 200,000 | Kept rows: 4,012
Processed rows: 250,000 | Kept rows: 4,951
Processed rows: 300,000 | Kept rows: 5,859
Processed rows: 350,000 | Kept rows: 6,863
Processed rows: 400,000 | Kept rows: 7,842
Processed rows: 450,000 | Kept rows: 8,923
Processed rows: 500,000 | Kept rows: 9,873
Processed rows: 550,000 | Kept rows: 10,998
Processed rows: 600,000 | Kept rows: 12,062
Processed rows: 650,000 | Kept rows: 12,980
Processed rows: 700,000 | Kept rows: 13,939
Processed rows: 750,000 | Kept rows: 14,891
Processed rows: 800,000 | Kept rows: 15,943
Processed rows: 850,000 | Kept rows: 16,960
Processed rows: 900,000 | Kept rows: 18,039
Processed rows: 950,000 | Kept rows: 19,093
Processed rows: 1,000,000 | Kep

In [4]:
# check result

output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
event_file = os.path.join(output_dir, "hdfs_v2_event_templates_v2.csv")

sample_df = pd.read_csv(event_file, nrows=10, encoding="utf-8-sig")
print(sample_df[["Content", "EventTemplate", "EventId", "BlockId"]].head(10))

                                             Content  \
0  Receiving BP-108841162-10.10.34.11-14400743609...   
1  src: /10.10.34.11:58869, dest: /10.10.34.11:50...   
2  PacketResponder: BP-108841162-10.10.34.11-1440...   
3  Scheduling blk_1074052640_311816 file /opt/hdf...   
4  Deleted BP-108841162-10.10.34.11-1440074360971...   
5  Receiving BP-108841162-10.10.34.11-14400743609...   
6  Received BP-108841162-10.10.34.11-144007436097...   
7  Receiving BP-108841162-10.10.34.11-14400743609...   
8  Received BP-108841162-10.10.34.11-144007436097...   
9  Receiving BP-108841162-10.10.34.11-14400743609...   

                                       EventTemplate   EventId         BlockId  
0  Receiving BP-<ID>-<IP>-<ID>:<BLOCK>_311816 src...  27135171  blk_1074052640  
1  src: /<IP_PORT>, dest: /<IP_PORT>, bytes: <ID>...  7ec375f1  blk_1074052640  
2  PacketResponder: BP-<ID>-<IP>-<ID>:<BLOCK>_311...  e1d721b8  blk_1074052640  
3  Scheduling <BLOCK>_311816 file <PATH><ID>-<IP>...  d26b6

In [6]:
import json
from sklearn.model_selection import train_test_split

# =====================================
# 1. Paths
# =====================================
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
input_file = os.path.join(output_dir, "hdfs_v2_event_templates_v2.csv")

sessions_file = os.path.join(output_dir, "hdfs_v2_sessions.csv")
train_sessions_file = os.path.join(output_dir, "hdfs_v2_train_sessions.csv")
test_sessions_file = os.path.join(output_dir, "hdfs_v2_test_sessions.csv")
train_txt_file = os.path.join(output_dir, "hdfs_v2_train.txt")
test_txt_file = os.path.join(output_dir, "hdfs_v2_test.txt")
vocab_file = os.path.join(output_dir, "hdfs_v2_event_vocab.json")


In [7]:

from collections import defaultdict
# =====================================
# 2. Read safely in chunks
# =====================================
chunk_size = 50000
sessions_dict = defaultdict(list)
unique_events = set()

total_rows = 0
valid_rows = 0

for chunk_num, chunk in enumerate(
    pd.read_csv(
        input_file,
        usecols=["BlockId", "EventId"],
        chunksize=chunk_size,
        encoding="utf-8-sig"
    ),
    start=1
):
    total_rows += len(chunk)

    chunk = chunk.dropna(subset=["BlockId", "EventId"]).copy()
    valid_rows += len(chunk)

    for block_id, event_id in zip(chunk["BlockId"].astype(str), chunk["EventId"].astype(str)):
        sessions_dict[block_id].append(event_id)
        unique_events.add(event_id)

    print(f"Chunk {chunk_num} done | Total rows: {total_rows:,} | Valid rows: {valid_rows:,} | Sessions: {len(sessions_dict):,}")


Chunk 1 done | Total rows: 50,000 | Valid rows: 50,000 | Sessions: 10,081
Chunk 2 done | Total rows: 100,000 | Valid rows: 100,000 | Sessions: 18,760
Chunk 3 done | Total rows: 150,000 | Valid rows: 150,000 | Sessions: 25,835
Chunk 4 done | Total rows: 200,000 | Valid rows: 200,000 | Sessions: 31,787
Chunk 5 done | Total rows: 250,000 | Valid rows: 250,000 | Sessions: 36,601
Chunk 6 done | Total rows: 300,000 | Valid rows: 300,000 | Sessions: 40,292
Chunk 7 done | Total rows: 350,000 | Valid rows: 350,000 | Sessions: 43,289
Chunk 8 done | Total rows: 400,000 | Valid rows: 400,000 | Sessions: 45,537
Chunk 9 done | Total rows: 450,000 | Valid rows: 450,000 | Sessions: 47,273
Chunk 10 done | Total rows: 500,000 | Valid rows: 500,000 | Sessions: 48,447
Chunk 11 done | Total rows: 550,000 | Valid rows: 550,000 | Sessions: 49,194
Chunk 12 done | Total rows: 600,000 | Valid rows: 600,000 | Sessions: 49,651
Chunk 13 done | Total rows: 650,000 | Valid rows: 650,000 | Sessions: 49,894
Chunk 14 d

In [8]:

# =====================================
# 3. Build session dataframe
# =====================================
session_rows = []
for block_id, event_list in sessions_dict.items():
    if len(event_list) >= 2:
        session_rows.append({
            "BlockId": block_id,
            "sequence_length": len(event_list),
            "event_sequence": " ".join(event_list)
        })

session_df = pd.DataFrame(session_rows)

print("\nTotal sessions kept for LogBERT:", len(session_df))
print(session_df["sequence_length"].describe())



Total sessions kept for LogBERT: 49998
count    49998.000000
mean        17.963539
std          4.833137
min          5.000000
25%         15.000000
50%         20.000000
75%         20.000000
max        301.000000
Name: sequence_length, dtype: float64


In [9]:

# =====================================
# 4. Save full sessions
# =====================================
session_df.to_csv(sessions_file, index=False, encoding="utf-8-sig")


In [10]:

# =====================================
# 5. Train/test split
# =====================================
train_df, test_df = train_test_split(session_df, test_size=0.2, random_state=42)

train_df.to_csv(train_sessions_file, index=False, encoding="utf-8-sig")
test_df.to_csv(test_sessions_file, index=False, encoding="utf-8-sig")


In [11]:

# =====================================
# 6. Save LogBERT txt files
# =====================================
with open(train_txt_file, "w", encoding="utf-8") as f:
    for line in train_df["event_sequence"]:
        f.write(line + "\n")

with open(test_txt_file, "w", encoding="utf-8") as f:
    for line in test_df["event_sequence"]:
        f.write(line + "\n")


In [12]:

# =====================================
# 7. Save vocab
# =====================================
event2idx = {event: idx + 1 for idx, event in enumerate(sorted(unique_events))}

with open(vocab_file, "w", encoding="utf-8") as f:
    json.dump(event2idx, f, indent=2)

print("\nDone.")
print("Saved:", sessions_file)
print("Saved:", train_sessions_file)
print("Saved:", test_sessions_file)
print("Saved:", train_txt_file)
print("Saved:", test_txt_file)
print("Saved:", vocab_file)
print("Vocabulary size:", len(event2idx))



Done.
Saved: D:\Downloads\Capstone\Project\logOutput_v2\hdfs_v2_sessions.csv
Saved: D:\Downloads\Capstone\Project\logOutput_v2\hdfs_v2_train_sessions.csv
Saved: D:\Downloads\Capstone\Project\logOutput_v2\hdfs_v2_test_sessions.csv
Saved: D:\Downloads\Capstone\Project\logOutput_v2\hdfs_v2_train.txt
Saved: D:\Downloads\Capstone\Project\logOutput_v2\hdfs_v2_test.txt
Saved: D:\Downloads\Capstone\Project\logOutput_v2\hdfs_v2_event_vocab.json
Vocabulary size: 507529


In [13]:
import os
import pandas as pd

output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
sessions_file = os.path.join(output_dir, "hdfs_v2_sessions.csv")

session_df = pd.read_csv(sessions_file, encoding="utf-8-sig")

print("Total sessions:", len(session_df))
print(session_df["sequence_length"].describe())
print("\nTop sequence lengths:")
print(session_df["sequence_length"].value_counts().head(20))

print("\nExample sessions:")
print(session_df.head(10))

Total sessions: 49998
count    49998.000000
mean        17.963539
std          4.833137
min          5.000000
25%         15.000000
50%         20.000000
75%         20.000000
max        301.000000
Name: sequence_length, dtype: float64

Top sequence lengths:
sequence_length
20    25027
15    16011
10     2407
9      1444
21     1005
27      541
12      463
26      368
22      337
16      289
33      244
18      225
32      166
23      136
5       135
13       90
29       89
39       86
31       71
19       69
Name: count, dtype: int64

Example sessions:
          BlockId  sequence_length  \
0  blk_1074052640               15   
1  blk_1073789757               26   
2  blk_1073789482               38   
3  blk_1073783334               29   
4  blk_1073867507               39   
5  blk_1073806022               33   
6  blk_1073927280               33   
7  blk_1073871135               33   
8  blk_1073800965               33   
9  blk_1073992994               30   

                     

In [15]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader

# =====================================
# 1. Paths
# =====================================
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
train_txt_file = os.path.join(output_dir, "hdfs_v2_train.txt")
test_txt_file = os.path.join(output_dir, "hdfs_v2_test.txt")
vocab_file = os.path.join(output_dir, "hdfs_v2_event_vocab.json")

# =====================================
# 2. Load vocabulary
# =====================================
with open(vocab_file, "r", encoding="utf-8") as f:
    event2idx = json.load(f)

print("Vocabulary size:", len(event2idx))

# =====================================
# 3. Config
# =====================================
MAX_LEN = 20        # most of your sessions are around 15-20
BATCH_SIZE = 32

# =====================================
# 4. Helper: encode one sequence
# =====================================
def encode_and_pad(seq_line, vocab, max_len=20):
    tokens = seq_line.strip().split()
    encoded = [vocab[token] for token in tokens if token in vocab]

    # truncate
    encoded = encoded[:max_len]

    # pad with 0
    if len(encoded) < max_len:
        encoded = encoded + [0] * (max_len - len(encoded))

    return encoded

# =====================================
# 5. Streaming Dataset
# =====================================
class LogTextDataset(Dataset):
    def __init__(self, file_path, vocab, max_len=20, max_samples=None):
        self.file_path = file_path
        self.vocab = vocab
        self.max_len = max_len
        self.sequences = []

        with open(file_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue

                self.sequences.append(encode_and_pad(line, vocab, max_len))

                if max_samples is not None and len(self.sequences) >= max_samples:
                    break

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long)

# =====================================
# 6. Start with a smaller subset first
#    Increase later after confirming it works
# =====================================
TRAIN_LIMIT = 100000
TEST_LIMIT = 20000

print("Loading train dataset...")
train_dataset = LogTextDataset(
    train_txt_file,
    event2idx,
    max_len=MAX_LEN,
    max_samples=TRAIN_LIMIT
)

print("Loading test dataset...")
test_dataset = LogTextDataset(
    test_txt_file,
    event2idx,
    max_len=MAX_LEN,
    max_samples=TEST_LIMIT
)

print("Train samples loaded:", len(train_dataset))
print("Test samples loaded:", len(test_dataset))

# =====================================
# 7. DataLoaders
# =====================================
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

# =====================================
# 8. Preview one batch
# =====================================
for batch in train_loader:
    print("Batch shape:", batch.shape)
    print(batch[:2])
    break

Vocabulary size: 507529
Loading train dataset...
Loading test dataset...
Train samples loaded: 39998
Test samples loaded: 10000
Train batches: 1250
Test batches: 313
Batch shape: torch.Size([32, 20])
tensor([[291106, 135231, 224835,   9106,  98931,   9106,  98931, 291106,  48122,
         224835, 291106, 189232, 234091,  98931, 291106, 189232, 291106, 168955,
         459998, 291106],
        [453871, 242983, 373783, 388740, 169707, 453871, 131151, 423711, 388740,
         169707, 453871, 144689, 373783, 388740, 169707, 272058, 246074, 246074,
         246074, 374086]])


In [16]:
# compact local vocab

import torch
from torch.utils.data import Dataset, DataLoader

# =====================================
# 1. Build compact vocab from loaded subset only
# =====================================
used_tokens = set()

for seq in train_dataset.sequences:
    used_tokens.update([x for x in seq if x != 0])

for seq in test_dataset.sequences:
    used_tokens.update([x for x in seq if x != 0])

used_tokens = sorted(list(used_tokens))
local_token2idx = {tok: i + 1 for i, tok in enumerate(used_tokens)}  # keep 0 for padding

print("Original huge vocab size:", len(event2idx))
print("Compact subset vocab size:", len(local_token2idx))

# =====================================
# 2. Remap sequences to compact IDs
# =====================================
def remap_sequence(seq, token_map):
    return [token_map.get(x, 0) if x != 0 else 0 for x in seq]

train_remapped = [remap_sequence(seq, local_token2idx) for seq in train_dataset.sequences]
test_remapped = [remap_sequence(seq, local_token2idx) for seq in test_dataset.sequences]

print("Example original sequence:")
print(train_dataset.sequences[0])
print("Example remapped sequence:")
print(train_remapped[0])

# =====================================
# 3. New dataset using compact vocab
# =====================================
class RemappedDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long)

train_dataset_small = RemappedDataset(train_remapped)
test_dataset_small = RemappedDataset(test_remapped)

train_loader_small = DataLoader(train_dataset_small, batch_size=32, shuffle=True)
test_loader_small = DataLoader(test_dataset_small, batch_size=32, shuffle=False)

print("Train batches:", len(train_loader_small))
print("Test batches:", len(test_loader_small))

# =====================================
# 4. Preview one safe compact batch
# =====================================
for batch in train_loader_small:
    print("Batch shape:", batch.shape)
    print(batch[:2])
    break

Original huge vocab size: 507529
Compact subset vocab size: 495970
Example original sequence:
[115056, 63425, 145019, 336393, 138389, 115056, 271120, 145019, 336393, 138389, 115056, 340909, 146399, 336393, 138389, 33304, 258196, 258196, 258196, 254455]
Example remapped sequence:
[112471, 62025, 141742, 328691, 135263, 112471, 264894, 141742, 328691, 135263, 112471, 333105, 143083, 328691, 135263, 32597, 252265, 252265, 252265, 248608]
Train batches: 1250
Test batches: 313
Batch shape: torch.Size([32, 20])
tensor([[429363,  74369, 215700, 204530, 207768, 429363,  87007, 270085, 204530,
         207768, 324250,  54293,  54293,  54293, 265991,      0,      0,      0,
              0,      0],
        [326848, 115795, 148048, 107201, 356905, 326848, 169177, 306410, 107201,
         356905, 186519, 443972, 443972, 443972, 252526,      0,      0,      0,
              0,      0]])


In [17]:
# reduce vocab size
import torch
import torch.nn as nn
import torch.optim as optim

# =====================================
# 1. Device
# =====================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =====================================
# 2. Small LogBERT-style model
# =====================================
class SimpleLogBERT(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, hidden_dim=128, num_layers=2, max_len=20):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()

        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)

        x_embed = self.embedding(x) + self.position_embedding(positions)
        x_trans = self.transformer(x_embed)
        logits = self.output_layer(x_trans)

        return logits

# =====================================
# 3. Masking function
# =====================================
def mask_input(batch, mask_token_id, pad_token_id=0, mask_prob=0.15):
    inputs = batch.clone()
    labels = batch.clone()

    rand = torch.rand(inputs.shape, device=inputs.device)
    mask = (rand < mask_prob) & (inputs != pad_token_id)

    inputs[mask] = mask_token_id
    labels[~mask] = -100   # ignore non-masked positions in loss

    return inputs, labels

# =====================================
# 4. Model setup
# =====================================
compact_vocab_size = len(local_token2idx) + 2   # +1 for padding(0), +1 for mask token
mask_token_id = len(local_token2idx) + 1

model = SimpleLogBERT(
    vocab_size=compact_vocab_size,
    embed_dim=64,
    num_heads=4,
    hidden_dim=128,
    num_layers=2,
    max_len=20
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("Compact vocab size for model:", compact_vocab_size)
print("Mask token id:", mask_token_id)

# =====================================
# 5. One quick training pass
# =====================================
model.train()

for batch_idx, batch in enumerate(train_loader_small):
    batch = batch.to(device)

    masked_input, labels = mask_input(batch, mask_token_id=mask_token_id)
    logits = model(masked_input)

    loss = criterion(logits.view(-1, compact_vocab_size), labels.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Batch {batch_idx + 1} | Loss: {loss.item():.4f}")

    # only test first few batches now
    if batch_idx == 4:
        break

print("Initial training step completed.")

Using device: cpu
Compact vocab size for model: 495972
Mask token id: 495971
Batch 1 | Loss: 13.2565
Batch 2 | Loss: 13.1858
Batch 3 | Loss: 13.3300
Batch 4 | Loss: 13.3523
Batch 5 | Loss: 13.3081
Initial training step completed.


In [19]:
# muti epoch training loop

import torch
import torch.nn as nn
import torch.optim as optim

# =====================================
# 1. Training settings
# =====================================
EPOCHS = 3
PRINT_EVERY = 200

# =====================================
# 2. Training loop
# =====================================
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    batch_count = 0

    for batch_idx, batch in enumerate(train_loader_small, start=1):
        batch = batch.to(device)

        masked_input, labels = mask_input(batch, mask_token_id=mask_token_id)
        logits = model(masked_input)

        loss = criterion(logits.view(-1, compact_vocab_size), labels.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

        if batch_idx % PRINT_EVERY == 0:
            avg_loss = total_loss / batch_count
            print(f"Epoch {epoch+1} | Batch {batch_idx} | Avg Loss: {avg_loss:.4f}")

    epoch_avg_loss = total_loss / batch_count
    print(f"\nEpoch {epoch+1} completed | Average Loss: {epoch_avg_loss:.4f}\n")

Epoch 1 | Batch 200 | Avg Loss: 13.2300
Epoch 1 | Batch 400 | Avg Loss: 13.2036
Epoch 1 | Batch 600 | Avg Loss: 13.1923
Epoch 1 | Batch 800 | Avg Loss: 13.1847
Epoch 1 | Batch 1000 | Avg Loss: 13.1867
Epoch 1 | Batch 1200 | Avg Loss: 13.1921

Epoch 1 completed | Average Loss: 13.1965

Epoch 2 | Batch 200 | Avg Loss: 13.1685
Epoch 2 | Batch 400 | Avg Loss: 13.1611
Epoch 2 | Batch 600 | Avg Loss: 13.1671
Epoch 2 | Batch 800 | Avg Loss: 13.1759
Epoch 2 | Batch 1000 | Avg Loss: 13.1851
Epoch 2 | Batch 1200 | Avg Loss: 13.1971

Epoch 2 completed | Average Loss: 13.2001

Epoch 3 | Batch 200 | Avg Loss: 13.2369
Epoch 3 | Batch 400 | Avg Loss: 13.2348
Epoch 3 | Batch 600 | Avg Loss: 13.2427
Epoch 3 | Batch 800 | Avg Loss: 13.2608
Epoch 3 | Batch 1000 | Avg Loss: 13.2739
Epoch 3 | Batch 1200 | Avg Loss: 13.2884

Epoch 3 completed | Average Loss: 13.2926



Saves previous data, restart point

In [22]:
import os
import json
import torch
from collections import Counter

# =====================================
# 1. Paths
# =====================================
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
train_txt_file = os.path.join(output_dir, "hdfs_v2_train.txt")
test_txt_file = os.path.join(output_dir, "hdfs_v2_test.txt")

state_dir = os.path.join(output_dir, "light_training_state")
os.makedirs(state_dir, exist_ok=True)

small_vocab_file = os.path.join(state_dir, "small_token2idx.json")
train_tensor_file = os.path.join(state_dir, "train_small.pt")
test_tensor_file = os.path.join(state_dir, "test_small.pt")
config_file = os.path.join(state_dir, "training_config.json")

# =====================================
# 2. Tuned settings for light workflow
# =====================================
MAX_VOCAB = 50000
MAX_LEN = 20
TRAIN_LIMIT = 100000
TEST_LIMIT = 20000

PAD_ID = 0
UNK_ID = 1
MASK_ID = 2

# =====================================
# 3. Pass 1: build small vocab from subset only
# =====================================
token_counter = Counter()

def update_counter_from_file(file_path, max_samples):
    count = 0
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens = line.split()[:MAX_LEN]
            token_counter.update(tokens)
            count += 1
            if count >= max_samples:
                break
    return count

train_count = update_counter_from_file(train_txt_file, TRAIN_LIMIT)
test_count = update_counter_from_file(test_txt_file, TEST_LIMIT)

most_common_tokens = [tok for tok, _ in token_counter.most_common(MAX_VOCAB)]
small_token2idx = {tok: i + 3 for i, tok in enumerate(most_common_tokens)}  # 0,1,2 reserved

print("Train lines used:", train_count)
print("Test lines used:", test_count)
print("Small vocab size:", len(small_token2idx) + 3)

# =====================================
# 4. Pass 2: encode + pad sequences
# =====================================
def encode_line(line, token_map, max_len=20):
    tokens = line.strip().split()[:max_len]
    encoded = [token_map.get(tok, UNK_ID) for tok in tokens]
    if len(encoded) < max_len:
        encoded += [PAD_ID] * (max_len - len(encoded))
    return encoded

def build_tensor_from_file(file_path, token_map, max_samples):
    sequences = []
    count = 0
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            sequences.append(encode_line(line, token_map, MAX_LEN))
            count += 1
            if count >= max_samples:
                break
    return torch.tensor(sequences, dtype=torch.long)

train_tensor = build_tensor_from_file(train_txt_file, small_token2idx, TRAIN_LIMIT)
test_tensor = build_tensor_from_file(test_txt_file, small_token2idx, TEST_LIMIT)

print("Train tensor shape:", tuple(train_tensor.shape))
print("Test tensor shape:", tuple(test_tensor.shape))

# =====================================
# 5. Save light state for future restarts
# =====================================
with open(small_vocab_file, "w", encoding="utf-8") as f:
    json.dump(small_token2idx, f)

torch.save(train_tensor, train_tensor_file)
torch.save(test_tensor, test_tensor_file)

training_config = {
    "PAD_ID": PAD_ID,
    "UNK_ID": UNK_ID,
    "MASK_ID": MASK_ID,
    "max_len": MAX_LEN,
    "batch_size": 16,
    "final_vocab_size": len(small_token2idx) + 3,
    "embed_dim": 32,
    "num_heads": 2,
    "hidden_dim": 64,
    "num_layers": 1,
    "train_limit": TRAIN_LIMIT,
    "test_limit": TEST_LIMIT,
    "max_vocab": MAX_VOCAB
}

with open(config_file, "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2)

print("\nSaved:", small_vocab_file)
print("Saved:", train_tensor_file)
print("Saved:", test_tensor_file)
print("Saved:", config_file)
print("Light training state is ready.")

Train lines used: 39998
Test lines used: 10000
Small vocab size: 50003
Train tensor shape: (39998, 20)
Test tensor shape: (10000, 20)

Saved: D:\Downloads\Capstone\Project\logOutput_v2\light_training_state\small_token2idx.json
Saved: D:\Downloads\Capstone\Project\logOutput_v2\light_training_state\train_small.pt
Saved: D:\Downloads\Capstone\Project\logOutput_v2\light_training_state\test_small.pt
Saved: D:\Downloads\Capstone\Project\logOutput_v2\light_training_state\training_config.json
Light training state is ready.


In [23]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# =====================================
# 1. Paths
# =====================================
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
state_dir = os.path.join(output_dir, "light_training_state")

train_tensor_file = os.path.join(state_dir, "train_small.pt")
test_tensor_file = os.path.join(state_dir, "test_small.pt")
config_file = os.path.join(state_dir, "training_config.json")
checkpoint_file = os.path.join(state_dir, "model_checkpoint_light.pt")

# =====================================
# 2. Load saved state
# =====================================
with open(config_file, "r", encoding="utf-8") as f:
    cfg = json.load(f)

train_tensor = torch.load(train_tensor_file)
test_tensor = torch.load(test_tensor_file)

PAD_ID = cfg["PAD_ID"]
UNK_ID = cfg["UNK_ID"]
MASK_ID = cfg["MASK_ID"]
MAX_LEN = cfg["max_len"]
BATCH_SIZE = cfg["batch_size"]
final_vocab_size = cfg["final_vocab_size"]

print("Train tensor shape:", tuple(train_tensor.shape))
print("Test tensor shape:", tuple(test_tensor.shape))
print("Final vocab size:", final_vocab_size)

# =====================================
# 3. Dataset and loaders
# =====================================
class SavedTensorDataset(Dataset):
    def __init__(self, tensor_data):
        self.tensor_data = tensor_data

    def __len__(self):
        return len(self.tensor_data)

    def __getitem__(self, idx):
        return self.tensor_data[idx]

train_dataset_final = SavedTensorDataset(train_tensor)
test_dataset_final = SavedTensorDataset(test_tensor)

train_loader_final = DataLoader(train_dataset_final, batch_size=BATCH_SIZE, shuffle=True)
test_loader_final = DataLoader(test_dataset_final, batch_size=BATCH_SIZE, shuffle=False)

# =====================================
# 4. Device
# =====================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =====================================
# 5. Model
# =====================================
class SimpleLogBERT(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_heads=2, hidden_dim=64, num_layers=1, max_len=20):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        x = self.embedding(x) + self.position_embedding(positions)
        x = self.transformer(x)
        return self.output_layer(x)

def mask_input(batch, mask_token_id=MASK_ID, pad_token_id=PAD_ID, mask_prob=0.10):
    inputs = batch.clone()
    labels = batch.clone()

    rand = torch.rand(inputs.shape, device=inputs.device)
    mask = (rand < mask_prob) & (inputs != pad_token_id)

    inputs[mask] = mask_token_id
    labels[~mask] = -100
    return inputs, labels

model = SimpleLogBERT(
    vocab_size=final_vocab_size,
    embed_dim=32,
    num_heads=2,
    hidden_dim=64,
    num_layers=1,
    max_len=MAX_LEN
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

# =====================================
# 6. Training
# =====================================
EPOCHS = 2
PRINT_EVERY = 100

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    batch_count = 0

    for batch_idx, batch in enumerate(train_loader_final, start=1):
        batch = batch.to(device)

        masked_input, labels = mask_input(batch)
        logits = model(masked_input)

        loss = criterion(logits.view(-1, final_vocab_size), labels.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

        if batch_idx % PRINT_EVERY == 0:
            avg_loss = total_loss / batch_count
            print(f"Epoch {epoch+1} | Batch {batch_idx} | Avg Loss: {avg_loss:.4f}")

    epoch_avg_loss = total_loss / batch_count
    print(f"\nEpoch {epoch+1} completed | Average Loss: {epoch_avg_loss:.4f}\n")

# =====================================
# 7. Test evaluation
# =====================================
model.eval()
total_test_loss = 0.0
test_batches = 0

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader_final, start=1):
        batch = batch.to(device)

        masked_input, labels = mask_input(batch)
        logits = model(masked_input)

        loss = criterion(logits.view(-1, final_vocab_size), labels.view(-1))

        total_test_loss += loss.item()
        test_batches += 1

avg_test_loss = total_test_loss / test_batches
print("Average test loss:", round(avg_test_loss, 4))

# =====================================
# 8. Save checkpoint
# =====================================
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": cfg,
    "final_vocab_size": final_vocab_size
}, checkpoint_file)

print("Saved checkpoint:", checkpoint_file)

Train tensor shape: (39998, 20)
Test tensor shape: (10000, 20)
Final vocab size: 50003
Using device: cpu
Epoch 1 | Batch 100 | Avg Loss: 8.8046
Epoch 1 | Batch 200 | Avg Loss: 7.4800
Epoch 1 | Batch 300 | Avg Loss: 6.3993
Epoch 1 | Batch 400 | Avg Loss: 5.6441
Epoch 1 | Batch 500 | Avg Loss: 5.1221
Epoch 1 | Batch 600 | Avg Loss: 4.6978
Epoch 1 | Batch 700 | Avg Loss: 4.4154
Epoch 1 | Batch 800 | Avg Loss: 4.2062
Epoch 1 | Batch 900 | Avg Loss: 4.0629
Epoch 1 | Batch 1000 | Avg Loss: 3.9481
Epoch 1 | Batch 1100 | Avg Loss: 3.8314
Epoch 1 | Batch 1200 | Avg Loss: 3.7310
Epoch 1 | Batch 1300 | Avg Loss: 3.6441
Epoch 1 | Batch 1400 | Avg Loss: 3.5770
Epoch 1 | Batch 1500 | Avg Loss: 3.5163
Epoch 1 | Batch 1600 | Avg Loss: 3.4564
Epoch 1 | Batch 1700 | Avg Loss: 3.4053
Epoch 1 | Batch 1800 | Avg Loss: 3.3628
Epoch 1 | Batch 1900 | Avg Loss: 3.3146
Epoch 1 | Batch 2000 | Avg Loss: 3.2781
Epoch 1 | Batch 2100 | Avg Loss: 3.2501
Epoch 1 | Batch 2200 | Avg Loss: 3.2109
Epoch 1 | Batch 2300 | A

In [25]:
# save trainig score

import os
import json

output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
state_dir = os.path.join(output_dir, "light_training_state")
metrics_file = os.path.join(state_dir, "training_metrics.json")

metrics = {
    "train_tensor_shape": [39998, 20],
    "test_tensor_shape": [10000, 20],
    "final_vocab_size": 50003,
    "device": "cpu",
    "epoch_1_avg_loss": 3.1549,
    "epoch_2_avg_loss": 2.5700,
    "avg_test_loss": 0.3874
}

with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", metrics_file)

Saved: D:\Downloads\Capstone\Project\logOutput_v2\light_training_state\training_metrics.json


In [26]:
import os
import json
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# =====================================
# 1. Paths
# =====================================
output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
state_dir = os.path.join(output_dir, "light_training_state")

test_tensor_file = os.path.join(state_dir, "test_small.pt")
config_file = os.path.join(state_dir, "training_config.json")
checkpoint_file = os.path.join(state_dir, "model_checkpoint_light.pt")
scores_file = os.path.join(state_dir, "test_anomaly_scores.csv")

# =====================================
# 2. Load config and test tensor
# =====================================
with open(config_file, "r", encoding="utf-8") as f:
    cfg = json.load(f)

test_tensor = torch.load(test_tensor_file)

PAD_ID = cfg["PAD_ID"]
MASK_ID = cfg["MASK_ID"]
MAX_LEN = cfg["max_len"]
final_vocab_size = cfg["final_vocab_size"]
BATCH_SIZE = cfg["batch_size"]

# =====================================
# 3. Dataset
# =====================================
class SavedTensorDataset(Dataset):
    def __init__(self, tensor_data):
        self.tensor_data = tensor_data
    def __len__(self):
        return len(self.tensor_data)
    def __getitem__(self, idx):
        return self.tensor_data[idx]

test_dataset = SavedTensorDataset(test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# =====================================
# 4. Model
# =====================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleLogBERT(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_heads=2, hidden_dim=64, num_layers=1, max_len=20):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_len, embed_dim, padding_idx=None)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        x = self.embedding(x) + self.position_embedding(positions)
        x = self.transformer(x)
        return self.output_layer(x)

def mask_input(batch, mask_token_id=MASK_ID, pad_token_id=PAD_ID, mask_prob=0.10):
    inputs = batch.clone()
    labels = batch.clone()
    rand = torch.rand(inputs.shape, device=inputs.device)
    mask = (rand < mask_prob) & (inputs != pad_token_id)
    inputs[mask] = mask_token_id
    labels[~mask] = -100
    return inputs, labels, mask

model = SimpleLogBERT(
    vocab_size=final_vocab_size,
    embed_dim=cfg["embed_dim"],
    num_heads=cfg["num_heads"],
    hidden_dim=cfg["hidden_dim"],
    num_layers=cfg["num_layers"],
    max_len=MAX_LEN
).to(device)

checkpoint = torch.load(checkpoint_file, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# =====================================
# 5. Per-sequence anomaly scores
# =====================================
loss_fn = nn.CrossEntropyLoss(ignore_index=-100, reduction="none")

rows = []
global_idx = 0

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)

        masked_input, labels, mask = mask_input(batch)
        logits = model(masked_input)

        token_losses = loss_fn(
            logits.view(-1, final_vocab_size),
            labels.view(-1)
        ).view(batch.size(0), batch.size(1))

        # average only over masked positions
        for i in range(batch.size(0)):
            masked_positions = mask[i]
            if masked_positions.sum().item() > 0:
                score = token_losses[i][masked_positions].mean().item()
            else:
                score = 0.0

            rows.append({
                "sample_index": global_idx,
                "anomaly_score": score
            })
            global_idx += 1

scores_df = pd.DataFrame(rows)
scores_df = scores_df.sort_values("anomaly_score", ascending=False).reset_index(drop=True)
scores_df.to_csv(scores_file, index=False)

print("Saved:", scores_file)
print(scores_df.head(20))

Saved: D:\Downloads\Capstone\Project\logOutput_v2\light_training_state\test_anomaly_scores.csv
    sample_index  anomaly_score
0           2658      15.533759
1            746      15.376625
2           7760      15.341509
3           8583      15.338066
4           4854      15.269183
5           1066      15.217255
6           7055      15.149442
7           2003      15.126602
8           8832      15.076033
9           4939      15.070385
10          2180      14.926853
11          3290      14.916364
12          6132      14.911743
13          2609      14.904015
14          8167      14.885357
15          6417      14.759959
16          4738      14.753231
17          1155      14.719855
18          1909      14.692359
19          8227      14.690817


In [27]:
# map anomalous score to actual event seq

import os
import pandas as pd

output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
state_dir = os.path.join(output_dir, "light_training_state")

scores_file = os.path.join(state_dir, "test_anomaly_scores.csv")
test_sessions_file = os.path.join(output_dir, "hdfs_v2_test_sessions.csv")
top_output_file = os.path.join(state_dir, "top_anomalous_sessions.csv")

scores_df = pd.read_csv(scores_file)
test_sessions_df = pd.read_csv(test_sessions_file, encoding="utf-8-sig")

# keep original order for indexing
test_sessions_df = test_sessions_df.reset_index().rename(columns={"index": "sample_index"})

merged_df = scores_df.merge(test_sessions_df, on="sample_index", how="left")

top_anomalies = merged_df.head(50)
top_anomalies.to_csv(top_output_file, index=False)

print("Saved:", top_output_file)
print(top_anomalies[["sample_index", "anomaly_score", "BlockId", "sequence_length", "event_sequence"]].head(10))

Saved: D:\Downloads\Capstone\Project\logOutput_v2\light_training_state\top_anomalous_sessions.csv
   sample_index  anomaly_score         BlockId  sequence_length  \
0          2658      15.533759  blk_1073828024               21   
1           746      15.376625  blk_1073962399               20   
2          7760      15.341509  blk_1074011359               21   
3          8583      15.338066  blk_1073856541               27   
4          4854      15.269183  blk_1073860934               50   
5          1066      15.217255  blk_1073951144               29   
6          7055      15.149442  blk_1074040448               27   
7          2003      15.126602  blk_1074012904               18   
8          8832      15.076033  blk_1073802853               27   
9          4939      15.070385  blk_1073896175               15   

                                      event_sequence  
0  e59be8cc 74d93a55 0f4b52ef 03832300 e59be8cc 9...  
1  86088057 2ba46f89 9439020a 2a537118 86088057 2...  

In [29]:
import os
import pandas as pd

output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
state_dir = os.path.join(output_dir, "light_training_state")

top_anomalies_file = os.path.join(state_dir, "top_anomalous_sessions.csv")
templates_file = os.path.join(output_dir, "hdfs_v2_event_templates_v2.csv")
readable_output_file = os.path.join(state_dir, "top_anomalous_sessions_readable.csv")

# load files
top_df = pd.read_csv(top_anomalies_file)
templates_df = pd.read_csv(templates_file, usecols=["EventId", "EventTemplate"], encoding="utf-8-sig")

# unique mapping from EventId to EventTemplate
event_map = templates_df.drop_duplicates(subset=["EventId"]).set_index("EventId")["EventTemplate"].to_dict()

# convert event_sequence into readable template sequence
def decode_sequence(event_sequence):
    event_ids = str(event_sequence).split()
    templates = [event_map.get(eid, f"[UNKNOWN:{eid}]") for eid in event_ids]
    return " | ".join(templates)

top_df["readable_sequence"] = top_df["event_sequence"].apply(decode_sequence)

# save
top_df.to_csv(readable_output_file, index=False)

print("Saved:", readable_output_file)
print(top_df[["sample_index", "anomaly_score", "BlockId", "sequence_length", "readable_sequence"]].head(5))

Saved: D:\Downloads\Capstone\Project\logOutput_v2\light_training_state\top_anomalous_sessions_readable.csv
   sample_index  anomaly_score         BlockId  sequence_length  \
0          2658      15.533759  blk_1073828024               21   
1           746      15.376625  blk_1073962399               20   
2          7760      15.341509  blk_1074011359               21   
3          8583      15.338066  blk_1073856541               27   
4          4854      15.269183  blk_1073860934               50   

                                   readable_sequence  
0  Receiving BP-<ID>-<IP>-<ID>:<BLOCK>_87200 src:...  
1  Receiving BP-<ID>-<IP>-<ID>:<BLOCK>_221575 src...  
2  Receiving BP-<ID>-<IP>-<ID>:<BLOCK>_270535 src...  
3  Receiving BP-<ID>-<IP>-<ID>:<BLOCK>_115717 src...  
4  Receiving BP-<ID>-<IP>-<ID>:<BLOCK>_120110 src...  


In [30]:
import os
import pandas as pd

output_dir = r"D:\Downloads\Capstone\Project\logOutput_v2"
state_dir = os.path.join(output_dir, "light_training_state")

readable_file = os.path.join(state_dir, "top_anomalous_sessions_readable.csv")
summary_file = os.path.join(state_dir, "top10_anomaly_summary.csv")

df = pd.read_csv(readable_file)

summary_df = df[[
    "sample_index",
    "anomaly_score",
    "BlockId",
    "sequence_length",
    "readable_sequence"
]].head(10).copy()

summary_df.to_csv(summary_file, index=False)

print("Saved:", summary_file)
print(summary_df)

Saved: D:\Downloads\Capstone\Project\logOutput_v2\light_training_state\top10_anomaly_summary.csv
   sample_index  anomaly_score         BlockId  sequence_length  \
0          2658      15.533759  blk_1073828024               21   
1           746      15.376625  blk_1073962399               20   
2          7760      15.341509  blk_1074011359               21   
3          8583      15.338066  blk_1073856541               27   
4          4854      15.269183  blk_1073860934               50   
5          1066      15.217255  blk_1073951144               29   
6          7055      15.149442  blk_1074040448               27   
7          2003      15.126602  blk_1074012904               18   
8          8832      15.076033  blk_1073802853               27   
9          4939      15.070385  blk_1073896175               15   

                                   readable_sequence  
0  Receiving BP-<ID>-<IP>-<ID>:<BLOCK>_87200 src:...  
1  Receiving BP-<ID>-<IP>-<ID>:<BLOCK>_221575 src...  
